# N004 · 函数、作用域与契约

**目标：** 把一个任务分解为参数清楚、返回值明确的函数。

**先修：** N003。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 位置与默认参数；局部作用域；类型标注；前置条件；可变默认参数风险。

**配套讲解来源：** [同名逐章意见](../../comment/004_functions_scope_contracts.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的算法载体是 LeetCode 1342"把数字减少到 0 的操作数"：给你一个非负整数 num，反复做下面两种操作直到它变成 0，返回总操作次数——如果当前数字是偶数，就把它除以 2；如果是奇数，就把它减 1。

具体例子：输入 `num=14`，输出 `6`。为什么是 6？我们一步步操作看：14 是偶数，除以 2 得 7（第 1 步）；7 是奇数，减 1 得 6（第 2 步）；6 是偶数，除以 2 得 3（第 3 步）；3 是奇数，减 1 得 2（第 4 步）；2 是偶数，除以 2 得 1（第 5 步）；1 是奇数，减 1 得 0（第 6 步）。数到 0 一共用掉 6 次操作，所以答案是 6。

但这一章真正的主角是**函数工程**：怎么把任务拆成参数清楚、返回值明确的小函数；为什么计数器必须是函数内部的局部变量（保证每次调用从零开始，不把上次的残留带进来）；以及函数的"契约"——入口要满足什么前置条件、出口保证返回什么。

先看本章要交付的两个接口：`is_even(num)` 负责判断奇偶，`number_of_steps(num)` 负责数步数。一个管"单步判断"，一个管"整个流程"，这就是"把一个任务分解为参数清楚、返回值明确的函数"（“学习目标”部分 目标原话）的具体形状。

## 2. 基础知识：先读懂这些词

- **函数（function）**：把一段逻辑打包成可重复调用的单位。本章有两个函数：`is_even` 负责"判断奇偶"这一件小事，`number_of_steps` 负责整个循环流程——这就是"任务分解"。
- **位置参数与默认参数**：调用时按位置对号入座的参数叫位置参数；定义时写成 `def f(a, b=0)` 的 b 带默认值，调用时不传就用默认值。本章代码没用到默认参数，但练习 2 会讲它的坑。
- **局部作用域（local scope）**：函数体内赋值过的变量只在函数内部存在，函数一返回就消失。比如 `steps` 只活在 `number_of_steps` 里面，外界看不见也改不着。这保证了两次调用互不干扰。
- **契约（contract）**：函数和使用者之间的约定，包含三部分——前置条件（调用前你必须满足什么，比如 num 必须非负）、后置条件（返回值保证是什么，比如"返回把 num 变 0 的操作次数"）、副作用（会不会改外部状态，本函数不会）。把契约想清楚，函数才不容易写错。
- **前置条件（precondition）**：调用方必须保证的输入条件。本章用 `if num < 0: raise ValueError(...)` 把"必须非负"这条前置条件显式写进代码，违反就直接报错。
- **类型标注（type annotation）**：`num: int -> int` 这种写法，声明"我打算接收整数、返回整数"。它是给人读的说明书，Python 运行时不强制。
- **循环不变量（loop invariant）**：循环每一轮都保持成立的性质。本章的不变量是：`steps` 恰好等于"已经执行的合法操作次数"，且 `num` 始终是非负整数。循环结束时 steps 就是总步数。
- **终止性（为什么循环一定会停）**：每次操作后 num 都严格变小（偶数除以 2 至少减半，奇数减 1 少 1），而 num 不能低于 0，所以它迟早会到 0，循环必然结束。
- **可变默认参数风险**：如果把默认值设成列表这类可变对象（如 `def f(acc=[])`），这个列表在所有调用之间是**共享同一个**的，上一次调用的残留会污染下一次。这是练习 2 的主题。
- **`while num:` 的真值写法**：在 Python 里 0 当作假、非零当作真，所以 `while num:` 等价于 `while num != 0:`，是"直到变成零"的紧凑写法。
- **无状态（stateless）**：函数的结果只由本次传入的参数决定，不依赖、也不改变任何"上次调用留下的痕迹"。`number_of_steps` 就是无状态的：连打三次，次次返回 6。
- **raise / 异常**：`raise ValueError('...')` 主动制造一个错误并带上说明文字，程序在这里中断，调用方要么处理这个异常、要么看着它崩。守卫 + raise 是"把契约写成代码"的直接手段。

## 3. 思路推导：从小例子开始

- **Step 1：把人话规则写成循环契约。** 题目规则本身就是"直到 num 变成 0 为止"的循环：每轮看 num 的奇偶，偶数就除 2，奇数就减 1。规则每一步都指定了唯一动作，不存在"选择歧义"，所以直接照抄即可。
- **Step 2：先拆出一个判断奇偶的小函数。** "是不是偶数"会被循环里用到，把它封装成 `is_even(num)` 返回 `num % 2 == 0`。函数小而单一，读主循环时一眼就懂。
- **Step 3：计数器放函数内部，从 0 起步。** `steps = 0` 定义在函数体内，所以它是局部变量——每次调用 `number_of_steps` 都会新建一个全新的 steps，绝不携带上次调用的记忆。这是本章强调的作用域要点。
- **Step 4：挡住负数。** 负数会让"减到 0"的逻辑失灵（奇数减 1 越来越负，永不归零），所以入口检查 `num < 0` 直接抛 ValueError，把前置条件写死在代码里。
- **Step 5：循环里用三元表达式一步完成"变新值"。** `num = num//2 if is_even(num) else num-1` 一行完成"偶除二、奇减一"，然后 `steps += 1`。
- **Step 6：循环结束后交答案。** `while num:` 变假（num 归零）时退出，返回 steps。整个函数从头到尾没有碰任何全局变量，出口就是干净的返回值。

**手算演示一轮（num=14，完整表格见 “运行示例”部分）**：“运行示例”部分 用 `show_table` 把 x=14 的全过程画成了表——当前值 14、动作"除2"、新值 7；当前值 7、动作"减1"、新值 6；当前值 6、动作"除2"、新值 3；当前值 3、动作"减1"、新值 2；当前值 2、动作"除2"、新值 1；当前值 1、动作"减1"、新值 0。表格一共 6 行（6 次动作），第 6 行之后 x 变成 0，`while x:` 条件为假，循环停。答案 6 与表格行数一致。建议你把这张表和 “分段实现”部分 的循环体逐行对照一遍：表格每一行就是 `num = num//2 if is_even(num) else num-1` 执行一次的快照，"动作"列告诉你这轮走的是哪个分支，"新值"列就是下一轮的 `while num:` 要检查的数。看懂这张表，你就看懂了这个函数的一生。

顺带注意 “运行示例”部分 的做法：它没有调用 `number_of_steps(14)`，而是把同样的规则重新手写了一遍循环来造表（`nxt=x//2 if is_even(x) else x-1`）。这样表格展示的是**过程**（每一步的中间状态），而函数只吐**最终步数**——想看过程就得把过程显式跑一遍，这是以后调试算法时"打日志还原现场"的雏形。

**作用域演示一轮**：第一次调用 `number_of_steps(14)` 返回 6 后，它内部的 steps 和 num 全部销毁；紧接着第二次调用 `number_of_steps(14)` 又新建 steps=0 从头数，仍然返回 6。假如你把 steps 写成全局变量，第二次调用会返回 12（把上次的 6 也累计进去）——这就是"计数器必须局部"的原因。

**再手算一轮（num=1，最短的正数路径）**：读入 num=1；`while num:` 里 1 是非零，为真，进入循环；`is_even(1)` 得 `1 % 2 == 0` 即 False，走 `num-1` 分支，num 变 0，steps 变 1；回到 `while num:`，0 为假，退出；返回 1。这一轮验证了"奇数减 1 直接归零"的最短路径，也验证了计数器只加了恰好一次。

**契约一句话复述**（建议你合上代码也能说出来）：入口契约是"num 必须是非负整数，否则抛 ValueError"；出口契约是"返回把 num 变成 0 所需的最少操作次数——本题每步动作被题意唯一指定，所以也就是操作次数"；副作用契约是"不改任何外部状态，重复调用结果一致"。

## 4. 核心代码：number_of_steps

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def number_of_steps(num: int) -> int:
    if num < 0:
        raise ValueError('num must be nonnegative')
    steps = 0
    while num:
        num = num // 2 if is_even(num) else num - 1
        steps += 1
    return steps
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

```python
def is_even(num: int) -> bool:
    return num % 2 == 0

def number_of_steps(num: int) -> int:
    if num < 0:
        raise ValueError('num must be nonnegative')
    steps = 0
    while num:
        num = num // 2 if is_even(num) else num - 1
        steps += 1
    return steps
```

**函数一：`is_even`**

- `def is_even(num: int) -> bool:` —— 定义"判断偶数"的小函数：接收整数，返回布尔值。它只做一件事，正是"任务分解"的最小示范。
- `return num % 2 == 0` —— num 除以 2 余 0 就是偶数。注意 `==` 比较的结果本身就是 `True` 或 `False`，直接返回即可，不需要写成 `if ...: return True else: return False` 的啰嗦形式。

**函数二：`number_of_steps`**

- `if num < 0:` / `raise ValueError('num must be nonnegative')` —— 前置条件守卫：负数不在契约范围内（负奇数减 1 会越来越小、永远到不了 0），所以直接抛出带说明文字的异常，宁可早失败也不给出错误答案。
- `steps = 0` —— 在函数体内初始化计数器为 0。关键在于它写在函数里面，因此是局部变量：每次调用都从 0 开始，函数结束后就销毁，上次调用的步数不会泄漏进来。
- `while num:` —— 循环条件。Python 里 0 相当于 False、非零相当于 True，所以这行的意思是"只要 num 还不是 0 就继续"。它把"直到变成零"这句人话原样翻译了出来。
- `num = num//2 if is_even(num) else num-1` —— 一行完成题意指定的转换：先调用 `is_even(num)` 判断奇偶，是偶数就取 `num//2`（整除，例如 14//2=7），是奇数就取 `num-1`。这里必须用 `//` 而不是 `/`：`/` 会得到 7.0 这样的浮点数，`//` 保持整数。
- `steps += 1` —— 每执行一次转换，计数加一。它和上一行合起来维持了循环不变量：steps 始终等于"到此为止执行的转换次数"。
- `return steps` —— num 变成 0、循环退出后，把累计的操作次数交出去。0 输入时循环体一次都不执行，直接返回 0。

**两个函数怎么配合着读**：先读 `is_even`（一行，契约是"给我一个整数，我还你一个布尔"），再读 `number_of_steps` 的骨架（守卫 → 初始化 → 循环 → 返回），最后才精读循环体那一行三元表达式。读代码先读契约、再读流程、最后抠细节，这个顺序能让你在以后面对几十行的函数时不迷路。

**两个常见错写，顺手对照一下**：

- 把 `num//2` 写成 `num/2`：`/` 是真除法，7/2 得 3.5，此后 num 变成浮点数一路小数下去，虽然多数情况也能停，但返回的 steps 会和整数语义的期望不一致（比如 14 走 `7.0 → 3.5 → 2.5 → 1.25 → 0.25 → …` 完全是另一个过程）。这提醒我们：整除用 `//`，别让类型悄悄漂移；
- 把 `steps = 0` 写到函数外面（全局）：表面上测试单次调用全过，但重复调用结果会累加。“自动测试”部分 那条 `[6,6,6]` 断言就是专门抓这种 bug 的——单次断言抓不到的错，要用"重复调用"才暴露得出来。

## 5. 分段实现：按顺序运行

**本章接口：** `number_of_steps(num)`、`is_even(num)`

### is_even

In [1]:
def is_even(num: int) -> bool:
    return num % 2 == 0

### number_of_steps

In [2]:
def number_of_steps(num: int) -> int:
    if num < 0:
        raise ValueError('num must be nonnegative')
    steps = 0
    while num:
        num = num // 2 if is_even(num) else num - 1
        steps += 1
    return steps

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

x=14
rows=[]
while x:
    nxt=x//2 if is_even(x) else x-1
    rows.append((x,'除2' if is_even(x) else '减1',nxt))
    x=nxt
show_table(['当前值','动作','新值'],rows)

当前值,动作,新值
14,除2,7
7,减1,6
6,除2,3
3,减1,2
2,除2,1
1,减1,0


## 7. 正确性、适用条件与复杂度

步骤计数等于已经执行的合法转换数。非负值严格下降保证终止；每一步是题意指定动作，不存在选择歧义。

**代价：** 单位字长模型下 O(log(num+1)) 时间、O(1) 辅助空间。

### 展开理解

**为什么是对的：** 我们盯住两条性质从头到尾成立。第一条：`steps` 每加一都对应一次真实发生的合法转换（除 2 或减 1），所以任何时刻 steps 都精确等于已经做的操作数；循环结束时 num 是 0，"做到 0"的过程每次操作都被数到且只数到一次，所以返回值就是总操作数。第二条：每个数是偶数还是奇数是确定的，题意为两种情况各指定了唯一的动作，所以每一步"该做什么"没有任何含糊，实现只是机械照做，正确性与题意逐条对应。

**拿 num=8 再手算一遍复杂度直觉**：8 是偶数除 2 得 4（1 步）、4 除 2 得 2（2 步）、2 除 2 得 1（3 步）、1 减 1 得 0（4 步）。数字 8→4→2→1→0，每除一次规模减半，4 步对应 log₂8=3 次除法加最后收尾的一次减 1。你可以把"步数 ≈ 二进制位数 + 1 的量级"当成快速估算口诀。

**为什么循环一定会停：** 每次操作后 num 都严格变小——偶数除以 2 显然变小（14 变 7），奇数减 1 也变小（7 变 6）。num 又不允许低于 0（减 1 只发生在正奇数身上，减完至少是 0；除 2 对非负偶数结果仍非负）。一个不断严格变小、又有下限（0）的整数序列，不可能无限走下去，迟早触到 0 让 `while num:` 变为假。所以这个循环保证终止，不会死循环。

**复杂度是多少：** 时间是 O(log(num+1))。直觉是：每遇到偶数就除以 2，数字的规模至少减半；奇数减 1 之后立刻变偶数又会被除 2。所以大约每两步数字就缩水一半，从 num 缩到 0 只需要"数字二进制位数"量级的步数。拿具体数字感受：num=14（二进制 1110，4 位）用 6 步；num=10⁹（约 30 位二进制）也只要 60 步左右，一瞬间就算完。辅助空间只有 steps 和 num 两个变量，是 O(1)。（这个 O(log) 是在"一次除法算一次操作"的字长模型下算的。）

**拿 num=8 再手算一遍复杂度直觉**：8 是偶数除 2 得 4（1 步）、4 除 2 得 2（2 步）、2 除 2 得 1（3 步）、1 减 1 得 0（4 步）。数字 8→4→2→1→0，每除一次规模减半，4 步对应 log₂8=3 次除法加最后收尾的一次减 1。你可以把"步数 ≈ 二进制位数 + 1 的量级"当成快速估算口诀。

**一个失效条件（对应 “掌握检查”部分 的自测要求）：** 改变一个输入约束——比如允许 num 为负——实现还成立吗？负奇数减 1 越来越小、负偶数整除会向负无穷取整（如 -4//2=-2 尚可，但 -1//2=-1 会卡死：-1 是奇数走减 1 变 -2，-2 除 2 变 -1，循环 -1→-2→-1 永不归零）。这正是入口守卫存在的意义：契约之外的行为不承诺，直接拒绝比假装正确诚实得多。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 一共四条断言，覆盖"正常值 / 零边界 / 最小正数 / 重复调用"四个层次。
“自动测试”部分 一共四行断言：

```python
assert number_of_steps(14) == 6

assert number_of_steps(0) == 0

assert number_of_steps(1) == 1

assert [number_of_steps(14) for _ in range(3)] == [6, 6, 6]

print('N004: 所有本章断言通过')
```

- `assert number_of_steps(14) == 6`：正常值测试。14 的完整过程是 14→7→6→3→2→1→0，共 6 步，同时覆盖了"除 2"和"减 1"两种动作，是主路径的全面检验。
- `assert number_of_steps(0) == 0`：零输入边界。num=0 时 `while num:` 一开始就是假，循环体不执行，直接返回 0——一步都不用做。它验证的是"循环条件用 0 当假值"和计数器初始值为 0。
- `assert number_of_steps(1) == 1`：最小正数边界。1 是奇数，减 1 得 0，恰好 1 步。它专门验证奇数分支在极端值上的行为。
- `assert [number_of_steps(14) for _ in range(3)] == [6,6,6]`：这条最有意思，它把同一个调用连做 3 次，期望三次都返回 6。它测的就是**局部作用域/无状态**这条契约：steps 是局部变量，函数不携带记忆，重复调用结果完全可复现。如果你把 steps 错写成全局变量，第二次调用会返回 12，这条立刻失败。
- 最后的 print：前面全部通过才会打印这句话，代表整组测试通过。

**故障推演：哪种写错会挂哪条？**

- 把 `num//2` 写成 `num-1`（两分支都减 1）：`number_of_steps(14)` 变成 14 ≠ 6，第一条挂；`number_of_steps(1)` 仍返回 1（单步路径一样），第二条反而绿——说明"过一条断言"远不等于"代码对"；
- 忘了 `steps += 1`：所有正数输入都返回 0，第 1、3、4 条全挂，只有 `number_of_steps(0) == 0` 碰巧绿——零输入断言天然测不到循环体内部，这是它的盲区；
- steps 写成全局变量：只有 `[6,6,6]` 那条挂（返回 `[6,12,18]`），前三条全绿——重复调用断言是唯一能抓住它的。

**断言分层一览表**：

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `number_of_steps(14) == 6` | 正常值 | 双分支混合的完整路径 |
| `number_of_steps(0) == 0` | 零边界 | 零输入零操作、不报错 |
| `number_of_steps(1) == 1` | 最小正数 | 奇数分支的极端值 |
| `[number_of_steps(14) for _ in range(3)] == [6,6,6]` | 无状态 | 局部变量、重复调用可复现 |

In [4]:
assert number_of_steps(14) == 6

assert number_of_steps(0) == 0

assert number_of_steps(1) == 1

assert [number_of_steps(14) for _ in range(3)] == [6, 6, 6]

print('N004: 所有本章断言通过')

N004: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 拆分奇偶处理但不增加无用包装。

**练习 2：** 修复可变默认参数共享状态。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（拆分奇偶处理但不增加无用包装）**：提示——本章已经有 `is_even` 这个恰到好处的拆分示范，你要做的是思考"拆到哪里为止"。可以尝试把"一步转换"本身抽成函数（比如 `step(num)` 返回转换后的新值），主循环里只写 `num = step(num); steps += 1`；但不要为了拆而拆出只被调用一次、名字不能自解释的函数。判断标准：每个函数是否能用一句话说清契约（输入什么、保证返回什么）。手算示例用 num=14，期望 6；边界用 0（期望 0）和 1（期望 1）。验收时把每个函数的契约各写一行注释放在函数头上，这一行注释就是"函数说明书"，写不出来的函数多半拆错了。
- **练习 2（修复可变默认参数共享状态）**：提示——先写一个错误示范：`def collect(x, acc=[]): acc.append(x); return acc`，然后连续调用 `collect(1)`、`collect(2)`，你会发现第二次返回 `[1, 2]` 而不是 `[2]`，因为默认的那个列表是函数定义时创建一次、所有调用共享的。修复方法是默认值改成不可变对象（如 `None`），函数体里再 `acc = [] if acc is None else acc` 新建列表。验证方式就是断言两次独立调用互不影响；边界情况包括"显式传入列表"和"完全不传"两种调用。做完和本章"steps 必须局部"对照一下，你会发现两者是同一个主题：**函数不能在调用之间留暗道**——区别只是全局变量是显式的暗道，可变默认参数是藏在签名里的暗道，后者更隐蔽、更容易在生产代码里发作。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def is_even(num: int) -> bool:
    return num % 2 == 0

def number_of_steps(num: int) -> int:
    if num < 0:
        raise ValueError('num must be nonnegative')
    steps = 0
    while num:
        num = num // 2 if is_even(num) else num - 1
        steps += 1
    return steps

# 示例与回归测试
assert number_of_steps(14) == 6

assert number_of_steps(0) == 0

assert number_of_steps(1) == 1

assert [number_of_steps(14) for _ in range(3)] == [6, 6, 6]

print('N004: 所有本章断言通过')

N004: 所有本章断言通过


## 11. 代表题与迁移

- **1342. Number of Steps to Reduce a Number to Zero**：就是本章 `number_of_steps` 的原题。它练的是本章的三个点：把人话规则直接翻译成 while 循环契约；用局部变量做计数器保证函数无状态（对应 `[6,6,6]` 那条断言）；以及负数守卫体现的前置条件意识（本题官方保证输入非负，守卫是教学加分项）。

**提交前的自检三问**：第一，我的循环条件是不是"直到变成 0"（`while num:`），终止有没有保证？第二，偶数分支用的是 `//` 还是会引入小数的 `/`？第三，函数里有没有任何跨调用残留的状态（全局变量、默认可变参数）？三问分别对应终止性、类型正确性和无状态契约。

**一个迁移预告**：这种"每步让数值严格变小、必然归零"的过程，本质上是按二进制位逐位消解；以后你遇到"按位运算"一类题（比如反复 `n & (n-1)` 数 1 的个数）时会再次看到同样的结构——本章的手算表格就是那时的思维底稿。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。